# Building an Offline RAG Pipeline Using PDF Chunking, FAISS, and Local Ollama

## 1. Project Introduction

This notebook demonstrates a **complete, 100% offline Retrieval-Augmented Generation (RAG) pipeline** running locally with **Ollama**, **LangChain**, and **FAISS** — requiring **no OpenAI or Azure OpenAI API keys**.

### Workflow Steps
1. **Load** multiple PDF documents with `PyPDFLoader`
2. **Extract** page text while preserving `source` filename and `page` number metadata
3. **Split** text into overlapping chunks (`chunk_size=1000`, `chunk_overlap=200`) using `RecursiveCharacterTextSplitter`
4. **Generate** dense vector embeddings locally with `OllamaEmbeddings` (`nomic-embed-text`)
5. **Store** embeddings and metadata in a persistent **FAISS** vector database
6. **Retrieve** the top 3 most semantically similar chunks for a user question
7. **Generate** grounded, source-cited answers locally with `ChatOllama` (`llama3.2`)

### Technology Stack
| Component | Technology |
|-----------|------------|
| Document Loading | LangChain `PyPDFLoader` (`pypdf`) |
| Text Splitting | `RecursiveCharacterTextSplitter` (1000 / 200) |
| Local Embeddings | `OllamaEmbeddings` (`nomic-embed-text`) |
| Vector Database | `FAISS` (`faiss-cpu`) |
| Local Chat LLM | `ChatOllama` (`llama3.2`) |
| Orchestration | LangChain (`langchain-ollama`, `langchain-community`) |

## 2. Import Libraries

Import the standard library, LangChain/Ollama components, and the project modules from `src/`.

In [1]:
# Standard library
import logging
import os
import sys
import time
from collections import Counter
from pathlib import Path

# Add project root to sys.path so src.* imports work
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Environment & LangChain / Ollama imports
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Project modules
from src.pdf_loader import load_pdfs_from_directory, display_document_summary
from src.text_chunker import (
    chunk_documents,
    calculate_chunk_statistics,
    display_chunk_statistics,
    display_sample_chunks,
)
from src.embeddings import (
    get_embeddings,
    validate_embedding_config,
    list_installed_ollama_models,
)
from src.vector_store import (
    create_vectorstore,
    load_vectorstore,
    get_or_create_vectorstore,
)
from src.retriever import retrieve_relevant_chunks, display_retrieved_chunks
from src.rag_pipeline import (
    get_llm,
    get_chat_model_name,
    answer_question,
    NOT_FOUND_MESSAGE,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

print('All libraries imported successfully (zero OpenAI/Azure dependencies)!')
print(f'  Python version : {sys.version.split()[0]}')
print(f'  Project root   : {PROJECT_ROOT}')

C:\Users\WELCOME\AppData\Local\Temp\ipykernel_4724\1806841033.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All libraries imported successfully (zero OpenAI/Azure dependencies)!
  Python version : 3.14.6
  Project root   : D:\Ass-1\rag-pdf-assistant


## 3. Load Configuration and Verify Local Ollama

We load optional settings from `.env` (falling back to offline Ollama defaults if `.env` is absent) and verify that the local Ollama daemon has both `nomic-embed-text` and `llama3.2` installed.

In [2]:
env_file = PROJECT_ROOT / '.env'
if env_file.exists():
    load_dotenv(dotenv_path=env_file)
    print(f'Loaded configuration from: {env_file}')
else:
    print('No .env file found — using built-in offline Ollama defaults.')

PDF_DIR = os.getenv('PDF_DIR', 'data/pdfs')
VECTORSTORE_PATH = os.getenv('VECTORSTORE_PATH', 'vectorstore')
CHUNK_SIZE = int(os.getenv('CHUNK_SIZE', 1000))
CHUNK_OVERLAP = int(os.getenv('CHUNK_OVERLAP', 200))
TOP_K = int(os.getenv('TOP_K_RESULTS', 3))
CHAT_MODEL = get_chat_model_name()

config = validate_embedding_config(check_server=True)
print('\nLocal Ollama Configuration:')
for k, v in config.items():
    print(f'  {k:20s}: {v}')
print(f'  {"chat_model":20s}: {CHAT_MODEL}')

if config.get('ollama_running'):
    installed = list_installed_ollama_models()
    print(f'  {"installed_models":20s}: {installed}')

print('\nPipeline Settings:')
print(f'  PDF Directory        : {PDF_DIR}')
print(f'  Vectorstore Path     : {VECTORSTORE_PATH}')
print(f'  Chunk Size           : {CHUNK_SIZE}')
print(f'  Chunk Overlap        : {CHUNK_OVERLAP}')
print(f'  Top-K Retrieval      : {TOP_K}')

Loaded configuration from: D:\Ass-1\rag-pdf-assistant\.env



Local Ollama Configuration:
  provider            : ollama
  model               : nomic-embed-text
  base_url            : http://localhost:11434
  api_key_required    : False
  ollama_running      : True
  model_installed     : True
  chat_model          : llama3.2


  installed_models    : ['nomic-embed-text:latest', 'llama3.2:latest', 'deepseek-r1:latest']

Pipeline Settings:
  PDF Directory        : data/pdfs
  Vectorstore Path     : vectorstore
  Chunk Size           : 1000
  Chunk Overlap        : 200
  Top-K Retrieval      : 3


## 4. Load PDF Documents

Load all PDF files from `data/pdfs/` using LangChain's `PyPDFLoader`, preserving the filename (`source`) and 1-indexed page number (`page`) as metadata.

In [3]:
pdf_dir_path = str(PROJECT_ROOT / PDF_DIR)
print(f'Loading PDFs from: {pdf_dir_path}')
print('-' * 60)

documents = load_pdfs_from_directory(pdf_dir_path)
print(f'\nSuccessfully loaded {len(documents)} page(s) from PDF documents.')

2026-11-22 12:06:47,197 [INFO] Found 3 PDF file(s) in 'D:\Ass-1\rag-pdf-assistant\data\pdfs'


2026-11-22 12:06:47,199 [INFO] Loading: 01_intro_to_ai.pdf


Loading PDFs from: D:\Ass-1\rag-pdf-assistant\data\pdfs
------------------------------------------------------------


2026-11-22 12:06:47,739 [INFO]   → Loaded 2 page(s) from '01_intro_to_ai.pdf'


2026-11-22 12:06:47,740 [INFO] Loading: 02_machine_learning.pdf


2026-11-22 12:06:47,761 [INFO]   → Loaded 2 page(s) from '02_machine_learning.pdf'


2026-11-22 12:06:47,763 [INFO] Loading: 03_deep_learning_rag.pdf


2026-11-22 12:06:47,787 [INFO]   → Loaded 2 page(s) from '03_deep_learning_rag.pdf'


2026-11-22 12:06:47,788 [INFO] 
Load summary: 6 total page(s) from 3 of 3 file(s).



Successfully loaded 6 page(s) from PDF documents.


## 5. Display Extracted PDF Text

Inspect the loaded documents, page counts, preserved metadata, and a text preview.

In [4]:
display_document_summary(documents)

print('--- Metadata & Text Preview of First Page ---')
print('Metadata:', documents[0].metadata)
print(documents[0].page_content[:500])
if len(documents[0].page_content) > 500:
    print('...')


LOADED DOCUMENTS SUMMARY
Total documents (files): 3
Total pages loaded     : 6
------------------------------------------------------------
  File : 01_intro_to_ai.pdf
  Pages: 2
  Preview: Introduction to Artificial Intelligence Chapter 1: What is Artificial Intelligence? Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems. These pro...

  File : 02_machine_learning.pdf
  Pages: 2
  Preview: Machine Learning Fundamentals Chapter 1: Introduction to Machine Learning Machine Learning (ML) is a subset of artificial intelligence that provides systems the ability to automatically learn and impr...

  File : 03_deep_learning_rag.pdf
  Pages: 2
  Preview: Deep Learning and Neural Networks Chapter 1: What is Deep Learning? Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers (hence 'deep') to progressiv...

--- Metadata & Text Preview of First Page ---
Metadata: {'producer': 'PyPDF', 'crea

## 6. Chunk PDF Content

Split all pages using `RecursiveCharacterTextSplitter` with `chunk_size=1000` and `chunk_overlap=200`.

In [5]:
print(f'Chunking {len(documents)} pages (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})...')
chunks = chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f'Created {len(chunks)} chunks.')

2026-11-22 12:06:47,831 [INFO] Chunking 6 page(s) with chunk_size=1000, chunk_overlap=200


2026-11-22 12:06:47,833 [INFO] Created 11 chunk(s).


Chunking 6 pages (chunk_size=1000, chunk_overlap=200)...
Created 11 chunks.


## 7. Display Sample Chunks

Verify that source filename and page number metadata are preserved on every chunk.

In [6]:
display_sample_chunks(chunks, n=3)


--- Sample Chunks (showing 3 of 11) ---

Chunk #1
  Source   : 01_intro_to_ai.pdf
  Page     : 1
  Size     : 935 chars
  Content  : Introduction to Artificial Intelligence Chapter 1: What is Artificial Intelligence? Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems. These processes include learning (the acquisition of information and rules for using the information), reason...

Chunk #2
  Source   : 01_intro_to_ai.pdf
  Page     : 1
  Size     : 801 chars
  Content  : called Strong AI). Narrow AI is designed to perform a specific task, such as facial recognition or internet searches. General AI, by contrast, is a system that can perform any intellectual task that a human being can. Current AI systems are all narrow AI, including even very powerful systems like GP...

Chunk #3
  Source   : 01_intro_to_ai.pdf
  Page     : 2
  Size     : 809 chars
  Content  : Chapter 4: Challenges and Ethics in AI The development of AI raises seve

## 8. Calculate Total Chunks

Count total chunks and view the per-PDF breakdown.

In [7]:
print(f'Total number of chunks: {len(chunks)}')
print('\nChunk breakdown by source PDF:')
source_counts = Counter(c.metadata.get('source', 'unknown') for c in chunks)
for source, count in sorted(source_counts.items()):
    print(f'  {source}: {count} chunks')

Total number of chunks: 11

Chunk breakdown by source PDF:
  01_intro_to_ai.pdf: 3 chunks
  02_machine_learning.pdf: 4 chunks
  03_deep_learning_rag.pdf: 4 chunks


## 9. Calculate Average Chunk Size

Compute accurate character-length statistics across all generated chunks.

In [8]:
stats = calculate_chunk_statistics(documents, chunks)
display_chunk_statistics(stats)

sizes = [len(c.page_content) for c in chunks]
print('\nDetailed size distribution:')
print(f'  Chunks under 500 chars  : {sum(1 for s in sizes if s < 500)}')
print(f'  Chunks 500-800 chars    : {sum(1 for s in sizes if 500 <= s < 800)}')
print(f'  Chunks 800-1000 chars   : {sum(1 for s in sizes if 800 <= s <= 1000)}')
print(f'  Chunks over 1000 chars  : {sum(1 for s in sizes if s > 1000)}')


CHUNKING STATISTICS
  PDF Documents      : 3
  Total Pages        : 6
  Total Chunks       : 11
  Avg Chunk Size     : 790.1 characters
  Min Chunk Size     : 345 characters
  Max Chunk Size     : 995 characters
  Chunk Size Setting : 1000
  Overlap Setting    : 200

Detailed size distribution:
  Chunks under 500 chars  : 1
  Chunks 500-800 chars    : 3
  Chunks 800-1000 chars   : 7
  Chunks over 1000 chars  : 0


## 10. Initialise Local Ollama Embeddings (`nomic-embed-text`)

We initialise `OllamaEmbeddings(model='nomic-embed-text')` and embed a test sentence to verify vector dimensionality locally.

In [9]:
embeddings = get_embeddings()
print(f'Embeddings initialised: {type(embeddings).__name__} (model={embeddings.model}, base_url={embeddings.base_url})')

test_text = 'Retrieval-Augmented Generation grounds LLM answers in PDF documents.'
test_vector = embeddings.embed_query(test_text)
EMBEDDING_DIM = len(test_vector)

print('\nTest embedding check:')
print(f'  Input text     : "{test_text}"')
print(f'  Vector dims    : {EMBEDDING_DIM}')
print(f'  First 5 values : {[round(v, 4) for v in test_vector[:5]]}')

2026-11-22 12:06:47,928 [INFO] Initialising OllamaEmbeddings: model='nomic-embed-text', base_url='http://localhost:11434'


Embeddings initialised: OllamaEmbeddings (model=nomic-embed-text, base_url=http://localhost:11434)


2026-11-22 12:06:50,041 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"



Test embedding check:
  Input text     : "Retrieval-Augmented Generation grounds LLM answers in PDF documents."
  Vector dims    : 768
  First 5 values : [0.0236, 0.0234, -0.1645, -0.0877, 0.0592]


## 11. Create FAISS Vectorstore

Embed all document chunks using local `nomic-embed-text` and store them along with their document/page metadata in FAISS.

In [10]:
vs_path = str(PROJECT_ROOT / VECTORSTORE_PATH)
pdf_dir_abs = str(PROJECT_ROOT / PDF_DIR)

t0 = time.time()
vectorstore = get_or_create_vectorstore(
    chunks=chunks,
    embeddings=embeddings,
    vectorstore_path=vs_path,
    pdf_dir=pdf_dir_abs,
    force_rebuild=False,
)
build_time = time.time() - t0

print(f'FAISS vectorstore ready in {build_time:.2f}s!')
print(f'  Type          : {type(vectorstore).__name__}')
print(f'  Index         : {type(vectorstore.index).__name__}')
print(f'  Total vectors : {vectorstore.index.ntotal}')
print(f'  Vector dims   : {vectorstore.index.d}')

2026-11-22 12:06:50,060 [INFO] PDF sources are unchanged — existing vectorstore is current.


2026-11-22 12:06:50,061 [INFO] Reusing existing vectorstore (PDFs unchanged).


2026-11-22 12:06:50,063 [INFO] Loading FAISS index from: D:\Ass-1\rag-pdf-assistant\vectorstore\faiss_index


2026-11-22 12:06:50,070 [INFO] Loading faiss with AVX2 support.


2026-11-22 12:06:50,071 [INFO] Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")


2026-11-22 12:06:50,073 [INFO] Loading faiss.


2026-11-22 12:06:50,111 [INFO] Successfully loaded faiss.


2026-11-22 12:06:50,129 [INFO] FAISS index loaded successfully.


FAISS vectorstore ready in 0.07s!
  Type          : FAISS
  Index         : IndexFlatL2
  Total vectors : 11
  Vector dims   : 768


## 12. Save and Load FAISS Index

Verify the persisted FAISS files on disk and reload the index in a fresh call without re-embedding.

In [11]:
index_path = Path(vs_path) / 'faiss_index'
print(f'Saved FAISS index directory: {index_path}')
for f in sorted(index_path.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1024:.1f} KB)')

print('\nReloading FAISS index from disk...')
vectorstore = load_vectorstore(vs_path, embeddings)
print(f'Reloaded successfully! Vectors in index: {vectorstore.index.ntotal}')

2026-11-22 12:06:50,141 [INFO] Loading FAISS index from: D:\Ass-1\rag-pdf-assistant\vectorstore\faiss_index


2026-11-22 12:06:50,143 [INFO] FAISS index loaded successfully.


Saved FAISS index directory: D:\Ass-1\rag-pdf-assistant\vectorstore\faiss_index
  index.faiss (33.0 KB)
  index.pkl (10.6 KB)

Reloading FAISS index from disk...
Reloaded successfully! Vectors in index: 11


## 13. Perform Semantic Search

Embed a user question with `nomic-embed-text` and query FAISS for the top 3 closest chunks.

In [12]:
sample_query = 'What is Artificial Intelligence?'
print(f'Query: "{sample_query}"')
print(f'Retrieving top {TOP_K} chunks...')

results = retrieve_relevant_chunks(sample_query, vectorstore, top_k=TOP_K)
print(f'Retrieved {len(results)} chunk(s).')

2026-11-22 12:06:50,152 [INFO] Retrieving top 3 chunks for query: 'What is Artificial Intelligence?…'


2026-11-22 12:06:50,176 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-11-22 12:06:50,178 [INFO] Retrieved 3 chunk(s).


Query: "What is Artificial Intelligence?"
Retrieving top 3 chunks...
Retrieved 3 chunk(s).


## 14. Display Top 3 Retrieved Chunks

Display source filename, page number, chunk index, L2 distance score (lower = closer), and content preview.

In [13]:
display_retrieved_chunks(results, show_full_content=False)


TOP 3 RETRIEVED CHUNKS

[Rank 1]
  Source     : 01_intro_to_ai.pdf
  Page       : 1
  Chunk #    : 0
  Similarity : 0.3375 (lower = more similar)
  Length     : 935 characters
  Preview    : Introduction to Artificial Intelligence Chapter 1: What is Artificial Intelligence? Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI was founded ...

[Rank 2]
  Source     : 01_intro_to_ai.pdf
  Page       : 1
  Chunk #    : 1
  Similarity : 0.5477 (lower = more similar)
  Length     : 801 characters
  Preview    : called Strong AI). Narrow AI is designed to perform a specific task, such as facial recognition or internet searches. General AI, by contrast, is a system that can perform any intellectual task that a human being can. Current AI systems

## 15. Generate Grounded Answer with Local `ChatOllama` (`llama3.2`)

Pass the top 3 retrieved chunks as context to local `ChatOllama(model='llama3.2', temperature=0)`. The system prompt strictly instructs the model to rely only on retrieved context and cite source document names and page numbers.

In [14]:
llm = get_llm()
print(f'Chat LLM initialised: {type(llm).__name__} (model={llm.model}, temperature={llm.temperature})')
print(f'Question: "{sample_query}"')

t0 = time.time()
single_result = answer_question(sample_query, vectorstore, top_k=TOP_K, llm=llm)
elapsed = time.time() - t0

print(f'\nGenerated in {elapsed:.2f}s')
print('=' * 60)
print(f'AI ANSWER ({llm.model})')
print('=' * 60)
print(single_result['answer'])
print('=' * 60)

2026-11-22 12:06:50,203 [INFO] Initialising ChatOllama: model='llama3.2', base_url='http://localhost:11434', temperature=0.0


2026-11-22 12:06:50,233 [INFO] Retrieving top 3 chunks for query: 'What is Artificial Intelligence?…'


Chat LLM initialised: ChatOllama (model=llama3.2, temperature=0.0)
Question: "What is Artificial Intelligence?"


2026-11-22 12:06:50,258 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-11-22 12:06:50,260 [INFO] Retrieved 3 chunk(s).


2026-11-22 12:06:52,427 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



Generated in 6.33s
AI ANSWER (llama3.2)
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by computer systems. These processes include learning, reasoning, and self-correction.

Sources: 01_intro_to_ai.pdf, Page 1


## 16. Test Multiple Sample Questions

Evaluate the pipeline across 4 in-domain questions plus 1 out-of-domain question (`What is quantum computing?`) to verify anti-hallucination behavior.

In [15]:
sample_questions = [
    'What is Artificial Intelligence?',
    'What are the types of machine learning?',
    'What is Retrieval-Augmented Generation (RAG)?',
    'What is backpropagation?',
    'What is quantum computing?',  # Out-of-domain: not in the PDFs
]

all_results = {}

for i, question in enumerate(sample_questions, 1):
    qt0 = time.time()
    r = answer_question(question, vectorstore, top_k=TOP_K, llm=llm)
    dt = time.time() - qt0
    all_results[question] = r
    print(f'\n{"=" * 60}')
    print(f'Q{i}: {question}  ({dt:.2f}s)')
    print('=' * 60)
    print(r['answer'])

2026-11-22 12:06:56,572 [INFO] Retrieving top 3 chunks for query: 'What is Artificial Intelligence?…'


2026-11-22 12:06:58,672 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-11-22 12:06:58,676 [INFO] Retrieved 3 chunk(s).


2026-11-22 12:06:58,780 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


2026-11-22 12:07:03,025 [INFO] Retrieving top 3 chunks for query: 'What are the types of machine learning?…'


2026-11-22 12:07:03,105 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


... (15 outputs left out)

2026-11-22 12:07:22,702 [INFO] Retrieved 3 chunk(s).



Q4: What is backpropagation?  (6.92s)
Backpropagation is the algorithm used to calculate gradients and update weights in a neural network, propagating the error signal from the output layer back through the network.

Sources: 03_deep_learning_rag.pdf, Page 1


2026-11-22 12:07:22,969 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



Q5: What is quantum computing?  (1.61s)
The answer was not found in the uploaded documents.


## 17. Display Source Document References

List the exact PDF filename, page number, and L2 distance for the chunks retrieved for every question.

In [16]:
print('SOURCE DOCUMENT REFERENCES')
print('=' * 60)

for question, r in all_results.items():
    print(f'\nQ: {question}')
    if NOT_FOUND_MESSAGE in r['answer']:
        print('   (Answer not found in uploaded documents — nearest chunks inspected below)')
    for src in r['sources']:
        print(f'   - {src["source"]} — Page {src["page"]}  (L2 distance: {src["score"]})')

SOURCE DOCUMENT REFERENCES

Q: What is Artificial Intelligence?
   - 01_intro_to_ai.pdf — Page 1  (L2 distance: 0.3375)
   - 01_intro_to_ai.pdf — Page 1  (L2 distance: 0.5477)
   - 01_intro_to_ai.pdf — Page 2  (L2 distance: 0.7245)

Q: What are the types of machine learning?
   - 02_machine_learning.pdf — Page 1  (L2 distance: 0.4786)
   - 02_machine_learning.pdf — Page 2  (L2 distance: 0.657)
   - 03_deep_learning_rag.pdf — Page 1  (L2 distance: 0.7051)

Q: What is Retrieval-Augmented Generation (RAG)?
   - 03_deep_learning_rag.pdf — Page 2  (L2 distance: 0.5284)
   - 03_deep_learning_rag.pdf — Page 2  (L2 distance: 0.7525)
   - 01_intro_to_ai.pdf — Page 1  (L2 distance: 0.9289)

Q: What is backpropagation?
   - 03_deep_learning_rag.pdf — Page 1  (L2 distance: 0.6416)
   - 03_deep_learning_rag.pdf — Page 1  (L2 distance: 0.8451)
   - 03_deep_learning_rag.pdf — Page 2  (L2 distance: 0.87)

Q: What is quantum computing?
   (Answer not found in uploaded documents — nearest chunks inspect

## 18. Final Observations and Metrics Summary

Consolidate all measured pipeline metrics from this actual local Ollama run.

In [17]:
print('=' * 60)
print('FINAL PIPELINE METRICS SUMMARY (ACTUAL EXECUTION)')
print('=' * 60)
print(f'  PDF Documents Processed  : {stats["num_documents"]}')
print(f'  Total Pages              : {stats["num_pages"]}')
print(f'  Total Chunks             : {stats["num_chunks"]}')
print(f'  Average Chunk Size       : {stats["avg_chunk_size"]} chars')
print(f'  Min / Max Chunk Size     : {stats["min_chunk_size"]} / {stats["max_chunk_size"]} chars')
print(f'  Chunk Size Setting       : {CHUNK_SIZE}')
print(f'  Chunk Overlap Setting    : {CHUNK_OVERLAP}')
print(f'  Embedding Model          : {embeddings.model} ({EMBEDDING_DIM}-d, local Ollama)')
print(f'  Chat LLM Model           : {llm.model} (local Ollama)')
print(f'  Embeddings Generated     : {len(chunks)}')
print(f'  FAISS Vectors Stored     : {vectorstore.index.ntotal}')
print(f'  Top-K Retrieved          : {TOP_K}')
print('=' * 60)

print('\nKey Observations:')
print('  1. The entire pipeline ran 100% offline via Ollama with zero cloud API keys.')
print('  2. nomic-embed-text embeddings produced accurate semantic retrieval in FAISS.')
print('  3. llama3.2 generated grounded answers with source PDF filenames and page numbers.')
print('  4. Unanswerable questions were handled without hallucination using strict context prompting.')
print('  5. FAISS persistence with content+model hashing prevents redundant re-embedding.')

FINAL PIPELINE METRICS SUMMARY (ACTUAL EXECUTION)
  PDF Documents Processed  : 3
  Total Pages              : 6
  Total Chunks             : 11
  Average Chunk Size       : 790.1 chars
  Min / Max Chunk Size     : 345 / 995 chars
  Chunk Size Setting       : 1000
  Chunk Overlap Setting    : 200
  Embedding Model          : nomic-embed-text (768-d, local Ollama)
  Chat LLM Model           : llama3.2 (local Ollama)
  Embeddings Generated     : 11
  FAISS Vectors Stored     : 11
  Top-K Retrieved          : 3

Key Observations:
  1. The entire pipeline ran 100% offline via Ollama with zero cloud API keys.
  2. nomic-embed-text embeddings produced accurate semantic retrieval in FAISS.
  3. llama3.2 generated grounded answers with source PDF filenames and page numbers.
  4. Unanswerable questions were handled without hallucination using strict context prompting.
  5. FAISS persistence with content+model hashing prevents redundant re-embedding.
